# 5. Evaluasi model terpilih

Model sudah disimpan oleh `train_model.py`. Notebook ini memuat ulang model dan mengukur
kinerjanya pada data uji. Kelas positif dalam perhitungan ini adalah phishing (label 0),
sehingga false negative berarti phishing yang salah dikenali sebagai legitimate.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "app").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import json
import joblib
import pandas as pd
from app.feature_extraction import FEATURE_NAMES
from train_model import evaluate

MODEL_DIR = ROOT / "models"
artikel = joblib.load(MODEL_DIR / "phishing_pipeline.joblib")
uji = pd.read_csv(ROOT / "dataset" / "processed" / "data_uji.csv")

X_uji = uji[FEATURE_NAMES]
y_uji = uji["ClassLabel"].astype(int)

metrik = evaluate(artikel["pipeline"], X_uji, y_uji)
print(pd.Series({k: v for k, v in metrik.items() if k != "confusion_matrix"}).round(5).to_string())

accuracy               0.99922
precision              0.99893
recall                 0.99982
f1                     0.99938
roc_auc                1.00000
false_negative_rate    0.00018


In [2]:
cm = metrik["confusion_matrix"]
print("Phishing terdeteksi sebagai phishing:", cm["true_phishing_predicted_phishing"])
print("Phishing terlewat:", cm["true_phishing_predicted_legitimate"])
print("Legitimate salah率达:", cm["true_legitimate_predicted_phishing"])
print("Legitimate benar:", cm["true_legitimate_predicted_legitimate"])

Phishing terdeteksi sebagai phishing: 5601
Phishing terlewat: 1
Legitimate salah率达: 6
Legitimate benar: 3338


In [3]:
metadata = json.loads((MODEL_DIR / "metadata.json").read_text(encoding="utf-8"))
print("Model:", metadata["model"])
print("Jumlah data uji:", metadata["pembagian_data"]["jumlah"]["test"])

Model: Random Forest
Jumlah data uji: 8946
